# 24. Agreement-gated Comparable after 17

23번 comparable sales 신호는 Fold2에서는 크게 먹혔지만 Fold3에서 깨졌습니다.

이번 실험은 comparable sales를 단독 보정으로 쓰지 않고, 17의 HistGB residual 방향과 같은 방향일 때만 제한적으로 적용합니다.

- 기준 예측: 17
- 추가 신호: comparable sales prediction
- 적용 조건: comparable gap과 17 residual-stack gap의 방향이 일치하는 row만
- 과도한 이동 방지: correction cap 적용
- OOF 기준 Fold2/Fold3 모두 개선될 때만 제출 파일 생성


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in candidates if (p / 'data').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

# 23번을 재현해 comparable cache와 17 OOF/test prediction을 가져옵니다.
# 23 자체는 안정 조건 미통과 시 제출 파일을 만들지 않습니다.
nb23 = json.loads((PROJECT_ROOT / 'notebooks' / '23_comparable_sales_after17.ipynb').read_text())
code23 = ''.join(nb23['cells'][1]['source'])
ns23 = {}
exec(code23, ns23)

rmse = ns23['rmse']
oof = ns23['oof'].copy()
folds = ns23['folds']
test = ns23['test']
sample_submission = ns23['sample_submission']
SUBMISSION_DIR = ns23['SUBMISSION_DIR']
test_frame = ns23['test_frame'].copy()
inner15 = ns23['inner15']
test_pred17 = ns23['test_pred17']
comparable_predict = ns23['comparable_predict']
comp_cache = ns23['comp_cache']
GATES = ns23['GATES']
train = ns23['train']

pred17_all = oof['Pred17'].to_numpy()
pred15_all = oof['Pred15'].to_numpy()
y_all = oof['Target'].to_numpy()
hist_gap_all = pred17_all - pred15_all
base17_rmse = rmse(y_all, pred17_all)
print(f'17 pooled RMSE: {base17_rmse:,.6f}')

ALPHAS = [0.02, 0.03, 0.05, 0.08]
CAPS = [300, 500, 800, 1200, 1800]
HIST_MIN_ABS = [0, 10, 25, 50]

rows = []
detail_tables = {}

for comp_key, comp_pred in comp_cache.items():
    scope, prior, top_k = comp_key
    comp_gap_all = comp_pred - pred17_all
    for gate_name, gate_col, gate_quantile in GATES:
        for alpha in ALPHAS:
            for cap in CAPS:
                for hist_min_abs in HIST_MIN_ABS:
                    pred24 = pred17_all.copy()
                    detail = []
                    offset = 0
                    for fold in folds:
                        frame = oof.loc[oof['Fold'] == fold].copy()
                        n = len(frame)
                        y = frame['Target'].to_numpy()
                        pred17 = frame['Pred17'].to_numpy()
                        comp_gap = comp_gap_all[offset:offset+n]
                        hist_gap = hist_gap_all[offset:offset+n]
                        if fold == 1:
                            mask = np.zeros(n, dtype=bool)
                        else:
                            agree = (comp_gap * hist_gap) > 0
                            strong_enough = np.abs(hist_gap) >= hist_min_abs
                            bounded = np.abs(comp_gap) <= cap * 6  # 극단 comparable outlier만 제거
                            if gate_col is None:
                                gate_mask = np.ones(n, dtype=bool)
                            else:
                                fit_hist = oof.loc[oof['Fold'] < fold].copy()
                                threshold = fit_hist[gate_col].quantile(gate_quantile)
                                gate_mask = frame[gate_col].to_numpy() <= threshold
                            mask = agree & strong_enough & bounded & gate_mask
                        clipped_gap = np.clip(comp_gap, -cap, cap)
                        pred_fold = pred17.copy()
                        pred_fold[mask] = np.clip(pred17[mask] + alpha * clipped_gap[mask], 0, None)
                        pred24[offset:offset+n] = pred_fold
                        detail.append({
                            'Fold': int(fold),
                            'Rows': n,
                            '17_RMSE': rmse(y, pred17),
                            '24_RMSE': rmse(y, pred_fold),
                            '24_vs_17_Improvement': rmse(y, pred17) - rmse(y, pred_fold),
                            'Applied_Rows': int(mask.sum()),
                            'Mean_abs_move': float(np.mean(np.abs(pred_fold - pred17))),
                        })
                        offset += n
                    detail = pd.DataFrame(detail)
                    eligible = detail.loc[detail['Fold'] > 1]
                    key = (scope, prior, top_k, gate_name, alpha, cap, hist_min_abs)
                    detail_tables[key] = detail
                    rows.append({
                        'Scope': scope,
                        'Prior': prior,
                        'TopK': top_k,
                        'Gate': gate_name,
                        'Alpha': alpha,
                        'Cap': cap,
                        'HistMinAbs': hist_min_abs,
                        '17_Pooled_RMSE': base17_rmse,
                        '24_Pooled_RMSE': rmse(y_all, pred24),
                        'Pooled_Improvement': base17_rmse - rmse(y_all, pred24),
                        'Improved_Eligible_Folds': int((eligible['24_vs_17_Improvement'] > 0).sum()),
                        'Worst_Eligible_Improvement': eligible['24_vs_17_Improvement'].min(),
                        'Fold2_Improvement': detail.loc[detail['Fold'] == 2, '24_vs_17_Improvement'].iloc[0],
                        'Fold3_Improvement': detail.loc[detail['Fold'] == 3, '24_vs_17_Improvement'].iloc[0],
                        'Mean_abs_move': detail['Mean_abs_move'].mean(),
                        'Applied_Rows_F2': detail.loc[detail['Fold'] == 2, 'Applied_Rows'].iloc[0],
                        'Applied_Rows_F3': detail.loc[detail['Fold'] == 3, 'Applied_Rows'].iloc[0],
                    })

summary = pd.DataFrame(rows).sort_values(
    ['Pooled_Improvement', 'Worst_Eligible_Improvement', 'Fold3_Improvement'],
    ascending=[False, False, False],
).reset_index(drop=True)
display(summary.head(40))

stable = summary.query(
    'Pooled_Improvement > 0.30 and Improved_Eligible_Folds == 2 and Worst_Eligible_Improvement > 0.05 and Fold3_Improvement > 0.05 and Mean_abs_move < 30 and Applied_Rows_F2 >= 10 and Applied_Rows_F3 >= 10'
).copy()

if len(stable) == 0:
    print('Agreement-gated comparable 보정이 안정성 조건을 통과하지 못했습니다. 제출 파일을 생성하지 않습니다.')
    best = summary.iloc[0].copy()
    CREATE_SUBMISSION = False
else:
    stable['OOF_Rank_Bucket'] = stable['Pooled_Improvement'].max() - stable['Pooled_Improvement'] <= 0.20
    pool = stable.loc[stable['OOF_Rank_Bucket']].copy() if stable['OOF_Rank_Bucket'].any() else stable.copy()
    best = pool.sort_values(['Mean_abs_move', 'Pooled_Improvement'], ascending=[True, False]).iloc[0].copy()
    CREATE_SUBMISSION = True
    print('선택된 24 agreement-gated comparable correction:')

display(best.to_frame('value'))
best_key = (
    best['Scope'], int(best['Prior']), int(best['TopK']), best['Gate'],
    float(best['Alpha']), int(best['Cap']), int(best['HistMinAbs'])
)
display(detail_tables[best_key])

leakage_audit = pd.Series({
    '17 baseline and comparable cache reproduced from leakage-safe notebook': True,
    'OOF comparable pools use only transactions earlier than validation months': True,
    'Agreement gate uses OOF Pred15/Pred17 only, not validation target': True,
    'Gate thresholds computed from earlier OOF folds only': True,
    'Scope/prior/top_k/gate/alpha/cap selected from Train OOF only': True,
    'No Test distribution/statistics used for selection': True,
    'Final Test comparable prediction uses full Train only after selection': True,
})
display(leakage_audit.rename('Passed'))
assert leakage_audit.all()

if CREATE_SUBMISSION:
    scope = best['Scope']
    prior = int(best['Prior'])
    top_k = int(best['TopK'])
    gate_name = best['Gate']
    alpha = float(best['Alpha'])
    cap = int(best['Cap'])
    hist_min_abs = int(best['HistMinAbs'])

    comp_test, _ = comparable_predict(train, test, scope=scope, prior=prior, top_k=top_k)
    comp_gap_test = comp_test - test_pred17
    hist_gap_test = test_pred17 - inner15['test_pred']
    agree_test = (comp_gap_test * hist_gap_test) > 0
    strong_test = np.abs(hist_gap_test) >= hist_min_abs
    bounded_test = np.abs(comp_gap_test) <= cap * 6
    gate_spec = next(g for g in GATES if g[0] == gate_name)
    _, gate_col, gate_quantile = gate_spec
    if gate_col is None:
        gate_mask_test = np.ones(len(test_frame), dtype=bool)
    else:
        threshold = oof[gate_col].quantile(gate_quantile)
        gate_mask_test = test_frame[gate_col].to_numpy() <= threshold
    mask_test = agree_test & strong_test & bounded_test & gate_mask_test
    clipped_gap_test = np.clip(comp_gap_test, -cap, cap)
    test_pred24 = test_pred17.copy()
    test_pred24[mask_test] = np.clip(test_pred17[mask_test] + alpha * clipped_gap_test[mask_test], 0, None)

    prediction_frame = pd.DataFrame({'ID': test['ID'], 'Target': test_pred24})
    submission = sample_submission[['ID']].merge(prediction_frame, on='ID', how='left', validate='one_to_one')
    assert submission['Target'].notna().all()
    assert len(submission) == len(sample_submission)
    assert sample_submission['ID'].equals(submission['ID'])
    output_path = SUBMISSION_DIR / '24_agreement_comparable_after17_submission.csv'
    submission.to_csv(output_path, index=False)
    print(f'Saved: {output_path}')
    display(submission.head())
else:
    output_path = None
    print('No submission saved for 24. Keep submitting 17_sklearn_residual_stack_submission.csv.')

result = {
    'create_submission': CREATE_SUBMISSION,
    'selected_scope': None if not CREATE_SUBMISSION else best['Scope'],
    'selected_prior': None if not CREATE_SUBMISSION else int(best['Prior']),
    'selected_top_k': None if not CREATE_SUBMISSION else int(best['TopK']),
    'selected_gate': None if not CREATE_SUBMISSION else best['Gate'],
    'selected_alpha': None if not CREATE_SUBMISSION else float(best['Alpha']),
    'selected_cap': None if not CREATE_SUBMISSION else int(best['Cap']),
    'selected_hist_min_abs': None if not CREATE_SUBMISSION else int(best['HistMinAbs']),
    'selected_oof_improvement': float(best['Pooled_Improvement']),
    'submission_path': None if output_path is None else str(output_path),
}
print(result)


Train: (1969, 11)
Train period: 202401 ~ 202512


,Fold,Train_rows,Valid_rows,04_RMSE,05_RMSE,06_RMSE,08_RMSE,09_RMSE,09_RMSLE,08_vs_06_Improvement,09_vs_08_Improvement,Local_Applied_Rows,Second_Local_Applied_Rows,Third_Local_Applied_Rows
0,Train-derived Fold 1,1201,278,"2,107.743326","2,107.743326","2,107.743326","2,107.743326","2,107.743326",0.055311,0.000000,0.000000,0,0,0
1,Train-derived Fold 2,1479,244,"2,634.763161","2,628.945780","2,621.229339","2,618.051538","2,614.984309",0.080279,3.177802,3.067229,141,103,108
2,Train-derived Fold 3,1723,246,"2,525.601387","2,522.504744","2,522.382035","2,520.144861","2,516.302312",0.071534,2.237174,3.842550,146,100,100


04 pooled RMSE: 2,420.08
05 pooled RMSE: 2,417.04
06 pooled RMSE: 2,414.33
08 pooled RMSE: 2,412.49
09 pooled RMSE: 2,410.15
Local-corrected improved folds: 2/2
Second-local improved folds: 2/2
Third-local improved folds: 2/2
08 vs 06 pooled improvement: 1.84
09 vs 08 pooled improvement: 2.34


Imputation statistics fitted on Train/fold-fit only                  True
OneHotEncoder fitted on Train/fold-fit only                          True
No independent dummy encoding on Test                                True
Fixed bins avoid validation/test distribution fitting                True
Target encoding excludes each training row target                    True
Residual model trained from time-OOF residuals only                  True
Local residual maps fitted from previous OOF residuals only          True
Second local correction selected with strict Train OOF thresholds    True
Third local correction selected with strict Train OOF thresholds     True
Validation uses strictly earlier transactions                        True
Weights, shrinkage and local corrections selected without Test       True
Test limited to transform and predict                                True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/09_residual08_cluster_correction_submission.csv


...

,Fold,Rows,17_RMSE,24_RMSE,24_vs_17_Improvement,Applied_Rows,Mean_abs_move
0,1,278,"2,107.743326","2,107.743326",0.000000,0,0.000000
1,2,244,"2,588.127758","2,584.877526",3.250231,34,17.465347
2,3,246,"2,511.258274","2,509.436106",1.822168,39,18.551120


17 baseline and comparable cache reproduced from leakage-safe notebook       True
OOF comparable pools use only transactions earlier than validation months    True
Agreement gate uses OOF Pred15/Pred17 only, not validation target            True
Gate thresholds computed from earlier OOF folds only                         True
Scope/prior/top_k/gate/alpha/cap selected from Train OOF only                True
No Test distribution/statistics used for selection                           True
Final Test comparable prediction uses full Train only after selection        True
Name: Passed, dtype: bool

Saved: /Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/24_agreement_comparable_after17_submission.csv


,ID,Target
0,TR_2427,"36,937.484718"
1,TR_0028,"48,170.602450"
2,TR_1461,"29,294.967192"
3,TR_1977,"47,532.510021"
4,TR_2351,"46,969.224666"


{'create_submission': True, 'selected_scope': 'dong', 'selected_prior': 120, 'selected_top_k': 40, 'selected_gate': 'low_std_q50', 'selected_alpha': 0.08, 'selected_cap': 1800, 'selected_hist_min_abs': 25, 'selected_oof_improvement': 1.7245450039831667, 'submission_path': '/Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/24_agreement_comparable_after17_submission.csv'}
